# Phase 3+4 integration — n=10, n_cues=3000, **no A_k** (Colab)

**Purpose.** Verify D1 (meta-stable rate) holds at the architecturally-intended Phase 3+4 integration regime at scale, before pivoting Phase 4 graduation to D1.

**Difference from saighi_n3k:** `--inhibition-gain 0.0` — A_k disabled. Report 036 closed the A_k investigation as orthogonal to the dominant substrate-shaping force; this run is the canonical non-A_k n=10 / n_cues=3000 baseline that we never had.

**Why this exists:**
- D1 currently has evidence only at Phase 4 *isolation* (frozen Phase 3c + synthetic drift, report 026 regime). It has never been measured at the integration regime where the cap_t_05 variance comes from.
- Checklist blocker F1 (multi-seed Phase 3+4 integration with active drift) is still ❌. This run closes F1 *and* establishes D1 at scale simultaneously.
- Existing n=10 (saighi) Drive data lacks per-query top_scores, so D1 cannot be recomputed from it. Fresh runs are required.

**Required local patch before launching:** `experiments/19_phase34_integrated.py` must include `meta_stable_w{2,3,4}` in `evaluate_combined()` payload. Committed in this branch; cell 1 verifies.

**Config:** n=10 seeds {17, 11, 23, 1, 2, 3, 5, 7, 13, 19}, n_cues=3000, st=0.3, online Hebbian, death on (thresh=0.05, window=10), A_k off (gain=0.0), no-reencode-discovered.

In [ ]:
# 1. Clone the repo and verify the meta-stable-rate patch is present.
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git log --oneline -5

# Patch sanity check: both the import AND the per-scale payload line must be present.
import subprocess
out = subprocess.check_output(['grep', '-c', 'meta_stable', 'experiments/19_phase34_integrated.py']).decode().strip()
assert int(out) >= 2, f'meta_stable_rate patch missing from exp 19 — abort! (matches={out})'
out2 = subprocess.check_output(['grep', '-c', 'meta_stable_w', 'experiments/19_phase34_integrated.py']).decode().strip()
assert int(out2) >= 1, f'per-scale payload line missing — abort! (matches={out2})'
print(f'meta_stable_rate patch present (meta_stable: {out} hits; meta_stable_w: {out2} hits)')

In [ ]:
# 2. Mount Drive and copy the Phase 3c codebook into place.
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
src = '/content/drive/MyDrive/neuro-ai/phase3c_codebook_reconstruction.pt'
dst_dir = 'reports/phase3c_reconstruction'
os.makedirs(dst_dir, exist_ok=True)
shutil.copy(src, f'{dst_dir}/phase3c_codebook_reconstruction.pt')
!ls -lh {dst_dir}/phase3c_codebook_reconstruction.pt

In [ ]:
# 3. Install deps.
!pip install -q datasets py-spy

In [ ]:
# 4. CPU-ONLY sanity check. Parent NEVER touches CUDA so workers can grab the GPU.
import os
print('cpu count:', os.cpu_count())

import sys; sys.path.insert(0, 'src')
from energy_memory.phase2.persistence import load_codebook
cb = load_codebook('reports/phase3c_reconstruction/phase3c_codebook_reconstruction.pt', device='cpu')
print('codebook (parent CPU load):', cb.shape, cb.dtype, cb.device)

# Verify meta_stable_rate is wired in evaluate_combined's payload (without running CUDA).
from energy_memory.phase2.metrics import meta_stable_rate
assert abs(meta_stable_rate([0.9, 0.99, 0.96, 0.92]) - 0.5) < 1e-9, 'meta_stable_rate behaving unexpectedly'
print('meta_stable_rate sanity ok:', meta_stable_rate([0.9, 0.99, 0.96, 0.92]))

del cb
import gc; gc.collect()

In [ ]:
# 4b. Pre-warm the wikitext cache so 10 subprocesses don't race on download.
print('warming wikitext cache...')
from energy_memory.phase2.corpus import load_corpus_splits
from pathlib import Path
splits = load_corpus_splits('wikitext', Path('.'), wikitext_name='wikitext-2-raw-v1')
print('  train:', len(splits['train']), 'rows')
print('  validation:', len(splits['validation']), 'rows')
print('cache warmed.')
del splits; gc.collect()

In [ ]:
# 4c. GPU info (no CUDA init).
print('=== GPU info ===')
!nvidia-smi --query-gpu=name,memory.total,compute_mode --format=csv
print()
print('=== Current GPU processes (should be empty before launching workers) ===')
!nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv

In [ ]:
# 5. PARALLEL launch. NOTE the only diff from saighi_n3k: --inhibition-gain 0.0 (A_k off).
PARALLEL_BATCH_SIZE = 10

import subprocess, os, time, signal
from pathlib import Path

os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
RUN_TAG = 'integration_n3k_noak'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
log_root = Path(f'reports/phase34_{RUN_TAG}_colab')
log_root.mkdir(parents=True, exist_ok=True)

def launch(seed):
    out_dir = f'reports/phase34_{RUN_TAG}_seed{seed}'
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    log_path = log_root / f'seed{seed}.log'
    logf = open(log_path, 'w')
    proc = subprocess.Popen(
        ['python', 'experiments/19_phase34_integrated.py',
         '--updater-kind', 'hebbian',
         '--device', 'cuda',
         '--seed', str(seed),
         '--n-cues', '3000',
         '--checkpoint-every', '500',
         '--success-threshold', '0.3',
         '--death-threshold', '0.05',
         '--death-window', '10',
         '--inhibition-gain', '0.0',
         '--inhibition-decay', '0.0',
         '--no-reencode-discovered',
         '--output-dir', out_dir],
        stdout=logf, stderr=subprocess.STDOUT,
    )
    return proc, logf, out_dir

def kill_all(procs_dict):
    for seed, (proc, logf, out_dir) in procs_dict.items():
        try:
            proc.send_signal(signal.SIGKILL)
            logf.close()
        except Exception:
            pass

def snapshot(procs_dict, t0):
    elapsed = (time.time() - t0) / 60
    print(f'  --- snapshot at {elapsed:.1f} min ---')
    try:
        gpu = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.used,utilization.gpu', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        print(f'  GPU: {gpu}')
        gpu_p = subprocess.check_output(
            ['nvidia-smi', '--query-compute-apps=pid,used_memory', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip()
        n_gpu_procs = len([l for l in gpu_p.splitlines() if l.strip()])
        print(f'  GPU processes attached: {n_gpu_procs}')
    except Exception as e:
        print(f'  GPU snapshot failed: {e}')
    for seed, (proc, logf, out_dir) in procs_dict.items():
        log = log_root / f'seed{seed}.log'
        size = log.stat().st_size if log.exists() else 0
        try:
            line = subprocess.check_output(['tail', '-1', str(log)],
                stderr=subprocess.DEVNULL).decode().strip()[:90]
        except Exception:
            line = ''
        print(f'  seed {seed:>2}: {size:>7}b  {line}')

def run_batch(seeds_batch, t0):
    procs = {seed: launch(seed) for seed in seeds_batch}
    print(f'  launched batch of {len(seeds_batch)}: {seeds_batch}  pids={[p[0].pid for p in procs.values()]}')
    remaining = dict(procs)
    snapshot_every = 3
    poll_count = 0
    try:
        while remaining:
            done_this_round = []
            for seed, (proc, logf, out_dir) in remaining.items():
                rc = proc.poll()
                if rc is not None:
                    logf.close()
                    elapsed = (time.time() - t0) / 60
                    ok = 'OK' if rc == 0 else f'FAILED (exit={rc})'
                    json_exists = Path(out_dir, 'phase34_results.json').exists()
                    print(f'  [{elapsed:5.1f} min] seed {seed:>2}: {ok}  json={json_exists}')
                    done_this_round.append(seed)
            for seed in done_this_round:
                del remaining[seed]
            if remaining:
                poll_count += 1
                if poll_count % snapshot_every == 0:
                    snapshot(remaining, t0)
                time.sleep(30)
    except KeyboardInterrupt:
        print('\n!!! Interrupted — SIGKILLing all remaining workers !!!')
        kill_all(remaining)
        raise

batches = [SEEDS[i:i+PARALLEL_BATCH_SIZE] for i in range(0, len(SEEDS), PARALLEL_BATCH_SIZE)]
print(f'Running {len(SEEDS)} seeds in {len(batches)} batches of up to {PARALLEL_BATCH_SIZE} ({time.strftime("%H:%M:%S")})')

t0 = time.time()
for i, batch in enumerate(batches, 1):
    print(f'\n=== Batch {i}/{len(batches)} ({time.strftime("%H:%M:%S")}) ===')
    run_batch(batch, t0)

print(f'\nALL DONE in {(time.time()-t0)/60:.1f} min')
print()
print('=== Final GPU state ===')
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

In [ ]:
# 5c. EMERGENCY: kill all workers if cell 5 misbehaves. Runtime → Interrupt cell 5 first, THEN run this.
import subprocess, signal, os
killed = 0
for line in subprocess.check_output(['ps', '-eo', 'pid,cmd']).decode().splitlines():
    if 'experiments/19' in line and 'grep' not in line:
        try:
            pid = int(line.split()[0])
            os.kill(pid, signal.SIGKILL)
            print(f'  killed {pid}')
            killed += 1
        except Exception as e:
            print(f'  {pid}: {e}')
print(f'killed {killed} workers')

In [ ]:
# 6. Copy results back to Drive.
import shutil, os
RUN_TAG = 'integration_n3k_noak'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
dst = '/content/drive/MyDrive/neuro-ai/results'
os.makedirs(dst, exist_ok=True)
for seed in SEEDS:
    src = f'reports/phase34_{RUN_TAG}_seed{seed}'
    if os.path.isdir(src):
        shutil.copytree(src, f'{dst}/phase34_{RUN_TAG}_seed{seed}', dirs_exist_ok=True)
shutil.copytree(f'reports/phase34_{RUN_TAG}_colab',
                f'{dst}/phase34_{RUN_TAG}_colab', dirs_exist_ok=True)
print('results in', dst)
!ls {dst} | grep -E 'integration_n3k_noak'

In [ ]:
# 7. In-notebook summary table — extended with D1 (meta_stable_w{2,3,4}).
import json
from pathlib import Path
import statistics as st

RUN_TAG = 'integration_n3k_noak'
SEEDS = [17, 11, 23, 1, 2, 3, 5, 7, 13, 19]
rows = []
header = (f'{"seed":>4} {"cond":>16} {"top1":>8} {"R@10":>8} {"cap_t05":>8} '
          f'{"ms_w2":>7} {"ms_w3":>7} {"ms_w4":>7} '
          f'{"cands":>6} {"cons":>5} {"deaths":>6}')
print(header)
for seed in SEEDS:
    p = Path(f'reports/phase34_{RUN_TAG}_seed{seed}/phase34_results.json')
    if not p.exists():
        print(f'{seed:>4} MISSING')
        continue
    d = json.loads(p.read_text())
    A = B = C = None
    for cond in ('baseline_static', 'phase3_reencode', 'phase3_phase4'):
        f = d['results'][cond][-1]
        cands = f.get('candidates_total', 0)
        cons_ = f.get('consolidations', 0)
        deaths = f.get('deaths_total', 0)
        ms2 = f.get('meta_stable_w2', float('nan'))
        ms3 = f.get('meta_stable_w3', float('nan'))
        ms4 = f.get('meta_stable_w4', float('nan'))
        print(f'{seed:>4} {cond:>16} {f["top1"]:>8.4f} {f["topk"]:>8.4f} {f["cap_t_05"]:>8.4f} '
              f'{ms2:>7.3f} {ms3:>7.3f} {ms4:>7.3f} '
              f'{cands:>6} {cons_:>5} {deaths:>6}')
        if cond == 'baseline_static':  A = f
        elif cond == 'phase3_reencode': B = f
        elif cond == 'phase3_phase4':   C = f
    if A and B and C:
        rows.append({'seed': seed,
                     'dR10_CA': C['topk']-A['topk'],
                     'dR10_CB': C['topk']-B['topk'],
                     'dtop1_CA': C['top1']-A['top1'],
                     'dcap_CA': C['cap_t_05']-A['cap_t_05'],
                     'dms2_CA': C.get('meta_stable_w2', 0.0) - A.get('meta_stable_w2', 0.0),
                     'dms3_CA': C.get('meta_stable_w3', 0.0) - A.get('meta_stable_w3', 0.0),
                     'dms4_CA': C.get('meta_stable_w4', 0.0) - A.get('meta_stable_w4', 0.0)})

print()
print(f'=== aggregate across {len(rows)} seeds ===')
tcrit = {9: 2.262, 8: 2.306, 7: 2.365, 6: 2.447, 5: 2.571, 4: 2.776}
df = max(1, len(rows)-1)
tc = tcrit.get(df, 2.262)
for k in ['dR10_CA','dR10_CB','dtop1_CA','dcap_CA','dms2_CA','dms3_CA','dms4_CA']:
    vals = [r[k] for r in rows]
    m = st.mean(vals); sd = st.stdev(vals) if len(vals)>1 else 0.0
    se = sd / (len(vals)**0.5) if len(vals)>1 else 0.0
    lo, hi = m - tc*se, m + tc*se
    pos = sum(1 for v in vals if v > 0)
    print(f'  {k}: mean={m:+.4f}, 95%CI [{lo:+.4f}, {hi:+.4f}], pos/total={pos}/{len(rows)}, '
          f'per-seed std={sd:.4f}')

# Persist aggregate for downstream report.
import json as _j
agg = {'seeds': SEEDS, 'rows': rows, 'tc_used': tc, 'df': df}
outp = Path(f'reports/phase34_{RUN_TAG}_aggregate.json')
outp.write_text(_j.dumps(agg, indent=2))
print(f'\naggregate saved -> {outp}')